# Fine-tune FunctionGemma 270M → Aloof App
**Goal:** Fine-tune `google/functiongemma-270m-it` on Hinglish WhatsApp messages  
**Output:** A ready-to-use `.task` file for the Aloof Android app  

### Before running:
1. Go to **Session options** (top right) → set **Accelerator = GPU T4 x2**
2. Add your HuggingFace token as a Kaggle secret named `HF_TOKEN`
   - Kaggle sidebar → **Add-ons → Secrets → Add new secret**
3. Accept the FunctionGemma license at: https://huggingface.co/google/functiongemma-270m-it

## Cell 1 — Install dependencies

In [ ]:
!pip install -q transformers trl peft accelerate datasets bitsandbytes
!pip install -q huggingface_hub
print('✅ Training deps installed')

## Cell 2 — Login to HuggingFace

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=hf_token)
print('✅ Logged in to HuggingFace')

## Cell 3 — Build training dataset (Hinglish WhatsApp messages)

In [ ]:
import json, random
from datasets import Dataset

SYSTEM = (
    'You are a reminder extraction assistant for a college student app.\n'
    'Analyze the WhatsApp message below and return ONLY a valid JSON object — no extra text.'
)
JSON_FORMAT = '''{
  "is_reminder": true or false,
  "category": one of [DEADLINE, ASSIGNMENT, EXAM, MEETING, REMINDER, SCHEDULE_CHANGE, HOLIDAY, OTHER],
  "date": "new date string or null (for SCHEDULE_CHANGE: the new date)",
  "original_date": "for SCHEDULE_CHANGE only: the old date being replaced, or null",
  "time": "extracted time string or null",
  "tags": ["tag1", "tag2"]
}'''

def make_text(message, result):
    prompt = (
        f'<start_of_turn>user\n'
        f'{SYSTEM}\n\n'
        f'Message: "{message}"\n\n'
        f'JSON format:\n{JSON_FORMAT}\n'
        f'<end_of_turn>\n'
        f'<start_of_turn>model'
    )
    completion = json.dumps(result, ensure_ascii=False)
    return prompt + '\n' + completion + '<end_of_turn>'

# ── Raw examples ──────────────────────────────────────────────────────────────
examples_raw = [
    # DEADLINE — College
    ('Assignment submission ki last date kal hai, 11:59 PM tak submit karo',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'kal', 'time': '11:59 PM', 'tags': ['assignment', 'submission', 'last date']}),
    ('Report submit karna hai by Friday EOD, late submissions accepted nahi honge',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Friday', 'time': None, 'tags': ['report', 'submit', 'Friday']}),
    ('Project deadline is today 5pm, dont miss it guys',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'today', 'time': '5pm', 'tags': ['project', 'deadline', 'today']}),
    ('Aaj raat 12 baje se pehle GitHub pe push kar dena',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'aaj raat', 'time': '12 baje', 'tags': ['deadline', 'GitHub']}),
    ('Case study submit karna hai Monday tak, no extensions',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Monday', 'time': None, 'tags': ['case study', 'submit', 'Monday']}),
    ('Literature review due tomorrow, midnight se pehle Moodle pe daalo',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'tomorrow', 'time': 'midnight', 'tags': ['literature review', 'due', 'Moodle']}),
    ('Internship form fill karna hai by 3pm today',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'today', 'time': '3pm', 'tags': ['internship', 'form', 'deadline']}),
    ('Fee submission last date is 25th June, baad mein late fine lagega',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': '25th June', 'time': None, 'tags': ['fee', 'submission', 'last date']}),
    ('Design doc submit karo by tomorrow 9am',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'tomorrow', 'time': '9am', 'tags': ['design doc', 'submit']}),
    ('Mini project report Friday tak submit karna mandatory hai',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Friday', 'time': None, 'tags': ['mini project', 'report', 'submit']}),

    # ASSIGNMENT — College
    ('DSA assignment ka 3rd question nahi hua kisi ka? Kal tak submit hai',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'kal', 'time': None, 'tags': ['DSA', 'assignment', 'submit']}),
    ('OS assignment group mein karo, Monday tak submit karna hai',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'Monday', 'time': None, 'tags': ['OS', 'assignment', 'group']}),
    ('Math homework complete karo, tomorrow class mein check hoga',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'tomorrow', 'time': None, 'tags': ['math', 'homework', 'class']}),
    ('Physics practical file Friday tak jama karni hai',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'Friday', 'time': None, 'tags': ['physics', 'practical', 'file']}),
    ('CN lab writeup submit karna hai next class se pehle',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': None, 'time': None, 'tags': ['CN', 'lab', 'writeup', 'submit']}),
    ('Accounts project individual hai, kal tak finish karo',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'kal', 'time': None, 'tags': ['accounts', 'project', 'individual']}),
    ('HR assignment 500 words ka hai, aaj raat tak submit karo on Teams',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'aaj raat', 'time': None, 'tags': ['HR', 'assignment', 'Teams']}),
    ('Coding assignment me bug hai toh fix karke resubmit karo by 6pm',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': None, 'time': '6pm', 'tags': ['coding', 'assignment', 'resubmit']}),
    ('Statistics worksheet pending hai, Tuesday tak submit karna hai compulsory',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'Tuesday', 'time': None, 'tags': ['statistics', 'worksheet', 'submit']}),
    ('ML assignment GitHub link bhejo professor ko by tomorrow',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'tomorrow', 'time': None, 'tags': ['ML', 'assignment', 'GitHub']}),

    # EXAM — College
    ('Kal DBMS ka internal hai, syllabus first 4 units tak hai',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'kal', 'time': None, 'tags': ['DBMS', 'internal', 'exam']}),
    ('Monday ko Networks ka midterm hai, MCQ format mein hoga',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'Monday', 'time': None, 'tags': ['Networks', 'midterm', 'MCQ']}),
    ('Viva Thursday hai, 10am se start hoga, prepared rehna',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'Thursday', 'time': '10am', 'tags': ['viva', 'Thursday']}),
    ('Practical exam next week hai, instruments khud laane honge',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'next week', 'time': None, 'tags': ['practical', 'exam']}),
    ('Quiz aaj 3rd lecture mein hoga, first 2 chapters se',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'aaj', 'time': None, 'tags': ['quiz', 'lecture', 'chapters']}),
    ('OS final exam 15th ko hai, full syllabus cover karna hai',
     {'is_reminder': True, 'category': 'EXAM', 'date': '15th', 'time': None, 'tags': ['OS', 'final', 'exam']}),
    ('Math test tomorrow 2nd period mein, integration and differentiation',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'tomorrow', 'time': None, 'tags': ['math', 'test', 'tomorrow']}),
    ('Kal chemistry paper hai 9am se, all the best sabko',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'kal', 'time': '9am', 'tags': ['chemistry', 'paper', 'exam']}),
    ('AI/ML unit test Friday ko 4th slot mein',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'Friday', 'time': None, 'tags': ['AI', 'ML', 'unit test']}),
    ('Viva aaj nahi kal hoga, list check karo apna slot',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'kal', 'time': None, 'tags': ['viva', 'slot']}),
    ('Mid sem exams 10th se 14th June tak hain, time table bheja hai',
     {'is_reminder': True, 'category': 'EXAM', 'date': '10th June', 'time': None, 'tags': ['mid sem', 'exams', 'timetable']}),

    # MEETING — College / Society
    ('Today seminar is at 3pm in room 201, attendance compulsory hai',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'today', 'time': '3pm', 'tags': ['seminar', 'attendance', 'room 201']}),
    ('Workshop on Python aaj 2-5pm tak hai',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'aaj', 'time': '2pm', 'tags': ['workshop', 'Python']}),
    ('Mentor meeting Tuesday 11am, come prepared with progress update',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'Tuesday', 'time': '11am', 'tags': ['mentor', 'meeting', 'progress']}),
    ('Extra class Saturday 9am mein hogi DSA ki',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'Saturday', 'time': '9am', 'tags': ['extra class', 'DSA', 'Saturday']}),
    ('Coding club ki meeting aaj 5pm mein hai, sab attend karo',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'aaj', 'time': '5pm', 'tags': ['coding club', 'meeting']}),
    ('NSS volunteers report karo Saturday 8am, cleanliness drive hai',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'Saturday', 'time': '8am', 'tags': ['NSS', 'volunteers', 'cleanliness drive']}),
    ('Photography club outing Sunday morning 7am, carry your equipment',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'Sunday', 'time': '7am', 'tags': ['photography', 'outing', 'equipment']}),
    ('Debate competition practice session Monday 3pm, mandatory for team',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'Monday', 'time': '3pm', 'tags': ['debate', 'practice', 'mandatory']}),
    ('Sports day practice Tuesday and Thursday 6am, attendance must',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'Tuesday', 'time': '6am', 'tags': ['sports day', 'practice', 'attendance']}),
    ('Library mein group study hai aaj 4pm, come on time',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'aaj', 'time': '4pm', 'tags': ['library', 'group study']}),

    # SCHEDULE_CHANGE — College / Society
    ('Kal ki Networks class cancel hai, sir bahar gaye hain',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': None, 'original_date': 'kal', 'time': None, 'tags': ['Networks', 'class', 'cancelled']}),
    ('Monday ki OS lecture postponed ho gayi, new date batayenge',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': None, 'original_date': 'Monday', 'time': None, 'tags': ['OS', 'lecture', 'postponed']}),
    ('Friday ka practical preponed to Thursday 2pm',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'Thursday', 'original_date': 'Friday', 'time': '2pm', 'tags': ['practical', 'preponed']}),
    ('Internal exam rescheduled, new date is 20th June',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': '20th June', 'original_date': None, 'time': None, 'tags': ['internal', 'exam', 'rescheduled']}),
    ('Aaj ki 3rd lecture shifted to 5th slot, room 204 se 301',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'aaj', 'original_date': 'aaj', 'time': None, 'tags': ['lecture', 'shifted', 'room change']}),
    ('Drama club rehearsal shifted to Friday 4pm instead of Thursday',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'Friday', 'original_date': 'Thursday', 'time': '4pm', 'tags': ['drama club', 'rehearsal', 'shifted']}),
    ('Kal class 8am ki jagah 10am se hogi, timing change',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'kal', 'original_date': 'kal', 'time': '10am', 'tags': ['class', 'timing', 'change']}),
    ('Aaj ki society meeting cancel hai, next Monday pe hogi',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'next Monday', 'original_date': 'aaj', 'time': None, 'tags': ['society', 'meeting', 'cancelled']}),

    "    # SCHEDULE_CHANGE — extra examples with original_date\n",
    "    ('Monday ka exam postponed ho gaya, ab 25th June ko hoga',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': '25th June', 'original_date': 'Monday', 'time': None, 'tags': ['exam', 'postponed']}),\n",
    "    ('Kal ka viva cancel hai, new date announce hongi',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': None, 'original_date': 'kal', 'time': None, 'tags': ['viva', 'cancelled']}),\n",
    "    ('Friday ki class ab Wednesday ko hogi 10am pe',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'Wednesday', 'original_date': 'Friday', 'time': '10am', 'tags': ['class', 'shifted']}),\n",
    "    ('Assignment deadline extend ho gayi, ab Monday tak submit karo',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'Monday', 'original_date': None, 'time': None, 'tags': ['assignment', 'deadline', 'extended']}),\n",
    "    ('Aaj ki lab cancel, Thursday ko extra lab hogi',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'Thursday', 'original_date': 'aaj', 'time': None, 'tags': ['lab', 'cancelled']}),\n",
    "    ('Workshop postponed indefinitely, will update',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': None, 'original_date': None, 'time': None, 'tags': ['workshop', 'postponed']}),\n",
    "    ('Submission date pushed to next Friday',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'next Friday', 'original_date': None, 'time': None, 'tags': ['submission', 'deadline']}),\n",
    "    ('Tuesday ki meeting Wednesday 3pm pe shifted ho gayi',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'Wednesday', 'original_date': 'Tuesday', 'time': '3pm', 'tags': ['meeting', 'shifted']}),\n",
    "    ('Internal 2 ka date change, ab 18th ko hoga instead of 15th',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': '18th', 'original_date': '15th', 'time': None, 'tags': ['internal', 'exam', 'rescheduled']}),\n",
    "    ('Sunday ki rehearsal cancel, next Sunday pe hogi',\n",
    "     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'next Sunday', 'original_date': 'Sunday', 'time': None, 'tags': ['rehearsal', 'cancelled']}),\n",
    "\n",
    "    # HOLIDAY\n",
    ('Kal Diwali ki wajah se college band hai, enjoy karo',
     {'is_reminder': False, 'category': 'HOLIDAY', 'date': 'kal', 'time': None, 'tags': ['Diwali', 'holiday', 'college']}),
    ('25th December ko Christmas holiday hai, no classes',
     {'is_reminder': False, 'category': 'HOLIDAY', 'date': '25th December', 'time': None, 'tags': ['Christmas', 'holiday']}),
    ('Republic day pe college closed rahega, 26 January',
     {'is_reminder': False, 'category': 'HOLIDAY', 'date': '26 January', 'time': None, 'tags': ['Republic Day', 'holiday', 'closed']}),
    ('Eid ke liye 2 din ki chhutti hai, 10th aur 11th ko',
     {'is_reminder': False, 'category': 'HOLIDAY', 'date': '10th', 'time': None, 'tags': ['Eid', 'holiday']}),
    ('Ganesh Chaturthi pe college band, 3 din ka break hai',
     {'is_reminder': False, 'category': 'HOLIDAY', 'date': None, 'time': None, 'tags': ['Ganesh Chaturthi', 'holiday', 'break']}),
    ('Holi mela mein jaana hai kal, rango ki taiyari karo',
     {'is_reminder': True, 'category': 'HOLIDAY', 'date': 'kal', 'time': None, 'tags': ['Holi', 'mela']}),
    ('Raksha Bandhan pe sab ghar aa rahe hain, 3rd August ko',
     {'is_reminder': True, 'category': 'HOLIDAY', 'date': '3rd August', 'time': None, 'tags': ['Raksha Bandhan', 'holiday']}),

    # REMINDER — Friends / Family
    ('Bhai kal plan hai na, mat bhool jaana 5 baje',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': '5 baje', 'tags': ['plan', 'reminder']}),
    ('Dont forget movie tonight 7pm, book your seat',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'tonight', 'time': '7pm', 'tags': ['movie', 'tonight']}),
    ('Kal birthday hai Rhea ka, surprise plan yaad hai na sabko?',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['birthday', 'surprise', 'plan']}),
    ('Gym jaana hai kal subah 6 baje, sab aana',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': '6 baje', 'tags': ['gym', 'subah']}),
    ('Beta kal doctor appointment hai 11am ka, mat bhoolna',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': '11am', 'tags': ['doctor', 'appointment']}),
    ('Dadi ka birthday is Sunday, sab ghar aana 6 baje',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'Sunday', 'time': '6 baje', 'tags': ['birthday', 'Sunday']}),
    ('Camping trip ke liye bags pack karke ready rehna Sunday subah 7am',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'Sunday', 'time': '7am', 'tags': ['camping', 'trip', 'bags']}),
    ('Tech fest mein participation ke liye forms aaj se available hain, fill karo',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': None, 'tags': ['tech fest', 'forms', 'participation']}),
    ('Hackathon team registration closes tonight at midnight',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'tonight', 'time': 'midnight', 'tags': ['hackathon', 'registration', 'deadline']}),
    ('Aaj shaam ko pooja hai 7 baje, sab ready rehna',
     {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': '7 baje', 'tags': ['pooja', 'shaam']}),

    # FORMAL ENGLISH
# FORMAL ENGLISH — Academic
    ('Please submit your assignment by Friday 11:59 PM. No late submissions will be accepted.',
     {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'Friday', 'original_date': None, 'time': '11:59 PM', 'tags': ['assignment', 'submission', 'deadline']}),
    ('Kindly note that the project report is due on Monday. Ensure it is uploaded to the portal.',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Monday', 'original_date': None, 'time': None, 'tags': ['project', 'report', 'deadline', 'portal']}),
    ('The mid-semester examination will be held on 15th June at 10:00 AM. Please carry your ID card.',
     {'is_reminder': True, 'category': 'EXAM', 'date': '15th June', 'original_date': None, 'time': '10:00 AM', 'tags': ['mid-semester', 'examination', 'ID card']}),
    ('A guest lecture is scheduled for tomorrow at 3 PM in the seminar hall. Attendance is compulsory.',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'tomorrow', 'original_date': None, 'time': '3 PM', 'tags': ['guest lecture', 'seminar', 'attendance']}),
    ('Dear students, the viva voce for the final year project will be conducted on Thursday at 11 AM.',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'Thursday', 'original_date': None, 'time': '11 AM', 'tags': ['viva', 'final year', 'project']}),
    ('Please be informed that the fee payment deadline is 20th June. Late payments will incur a fine.',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': '20th June', 'original_date': None, 'time': None, 'tags': ['fee', 'payment', 'deadline']}),
    ('The internal assessment for Operating Systems is scheduled for next Monday. Syllabus: Units 1-4.',
     {'is_reminder': True, 'category': 'EXAM', 'date': 'next Monday', 'original_date': None, 'time': None, 'tags': ['internal', 'Operating Systems', 'assessment']}),
    ('Reminder: Lab records must be submitted before the practical examination on Wednesday.',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Wednesday', 'original_date': None, 'time': None, 'tags': ['lab', 'records', 'practical', 'submission']}),
    ('All students are required to attend the orientation session on Saturday at 9 AM.',
     {'is_reminder': True, 'category': 'MEETING', 'date': 'Saturday', 'original_date': None, 'time': '9 AM', 'tags': ['orientation', 'session', 'attendance']}),
    ('The hackathon registration closes tonight at midnight. Teams of 3-4 members.',
     {'is_reminder': True, 'category': 'DEADLINE', 'date': 'tonight', 'original_date': None, 'time': 'midnight', 'tags': ['hackathon', 'registration', 'deadline']}),

    # FORMAL ENGLISH — Schedule changes
    ('Please note that tomorrow's lecture has been cancelled due to the faculty's absence.',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': None, 'original_date': 'tomorrow', 'time': None, 'tags': ['lecture', 'cancelled']}),
    ('The examination originally scheduled for Monday has been rescheduled to Wednesday, 20th June.',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': '20th June', 'original_date': 'Monday', 'time': None, 'tags': ['examination', 'rescheduled']}),
    ('Kindly note that Friday's practical session has been preponed to Thursday at 2 PM.',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'Thursday', 'original_date': 'Friday', 'time': '2 PM', 'tags': ['practical', 'preponed']}),
    ('The submission deadline has been extended to next Monday. Please plan accordingly.',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': 'next Monday', 'original_date': None, 'time': None, 'tags': ['submission', 'deadline', 'extended']}),
    ('The seminar scheduled for this afternoon stands cancelled. A new date will be announced shortly.',
     {'is_reminder': True, 'category': 'SCHEDULE_CHANGE', 'date': None, 'original_date': 'today', 'time': None, 'tags': ['seminar', 'cancelled']}),

    # FORMAL ENGLISH — Personal
    ('Your dental appointment is confirmed for tomorrow at 11:30 AM. Please arrive 10 minutes early.',
     {'is_reminder': True, 'category': 'PERSONAL', 'date': 'tomorrow', 'original_date': None, 'time': '11:30 AM', 'tags': ['dental', 'appointment']}),
    ('Please remember to pay the electricity bill before 25th to avoid disconnection.',
     {'is_reminder': True, 'category': 'PERSONAL', 'date': '25th', 'original_date': None, 'time': None, 'tags': ['electricity', 'bill', 'payment']}),
    ('Kindly collect your passport from the regional office by Friday.',
     {'is_reminder': True, 'category': 'PERSONAL', 'date': 'Friday', 'original_date': None, 'time': None, 'tags': ['passport', 'collect']}),

    # FORMAL ENGLISH — Event / Info
    ('Please be informed that the college will remain closed on 26th January on account of Republic Day.',
     {'is_reminder': False, 'category': 'EVENT', 'date': '26th January', 'original_date': None, 'time': None, 'tags': ['Republic Day', 'holiday', 'closed']}),
    ('The results for the mid-semester examination have been declared. Kindly check the portal.',
     {'is_reminder': False, 'category': 'INFO', 'date': None, 'original_date': None, 'time': None, 'tags': ['results', 'mid-semester', 'portal']}),
    ('This is to inform all students that the revised timetable has been uploaded to the ERP portal.',
     {'is_reminder': False, 'category': 'INFO', 'date': None, 'original_date': None, 'time': None, 'tags': ['timetable', 'revised', 'portal']}),
    ('College will be closed from 10th to 14th October on account of Dussehra and Diwali holidays.',
     {'is_reminder': False, 'category': 'EVENT', 'date': '10th October', 'original_date': None, 'time': None, 'tags': ['Dussehra', 'Diwali', 'holiday', 'closed']}),

    # FORMAL ENGLISH — Noise / Other
    ('Thank you all for your participation in the event. It was a great success.',
     {'is_reminder': False, 'category': 'OTHER', 'date': None, 'original_date': None, 'time': None, 'tags': []}),
    ('Congratulations to the winners of the inter-college coding competition!',
     {'is_reminder': False, 'category': 'OTHER', 'date': None, 'original_date': None, 'time': None, 'tags': []}),
    ('Please find the meeting minutes attached for your reference.',
     {'is_reminder': False, 'category': 'OTHER', 'date': None, 'original_date': None, 'time': None, 'tags': []}),
    ('The attendance sheet for last week has been updated on the portal.',
     {'is_reminder': False, 'category': 'INFO', 'date': None, 'original_date': None, 'time': None, 'tags': ['attendance', 'portal']}),

        # NOISE — NOT reminders
    ('Bhai woh meme dekha?? dying', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Kya kha raha hai aaj?', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Netflix pe kya dekh rahe ho aajkal?', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Bhai score dekha? India toh haar gayi', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Haha lmao thats so random', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Sab theek hai? Kaafi dino se baat nahi hui', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Great event tha yesterday, sab log mast the', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Congrats to all winners!', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Good morning sabko', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Beta khana kha liya? Ghar kab aa raha hai?', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Ok', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Noted', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Sahi hai bro', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Koi online hai?', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Arey wah! Congrats', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Mujhe neend aa rahi hai yaar', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Bhai seriously?', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Aur bata kya scene hai', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Professor ne kya bola?', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),
    ('Good night everyone', {'is_reminder': False, 'category': 'OTHER', 'date': None, 'time': None, 'tags': []}),

    # CASUAL COMMANDS — kardena / bhardena forms
    ('Gas ka bill bhardena', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['gas', 'bill']}),
    ('Light ka bill bhardena kal tak', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['light', 'bill']}),
    ('Rent bhardena is mahine', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'is mahine', 'time': None, 'tags': ['rent', 'bill']}),
    ('Form bhardena aaj', {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': None, 'tags': ['form']}),
    ('Fee bhardena Monday tak', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Monday', 'time': None, 'tags': ['fee']}),
    ('Medicine le aana aur bill bhardena', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['medicine', 'bill']}),
    ('Assignment submit kardena aaj raat tak', {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'aaj raat', 'time': None, 'tags': ['assignment', 'submit']}),
    ('Professor ko mail kardena kal', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['professor', 'mail']}),
    ('Library book return kardena Friday tak', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Friday', 'time': None, 'tags': ['library', 'book', 'return']}),
    ('Form download kardena aur fill kardena', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['form', 'download']}),
    ('Registration kardena aaj, last date hai', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'aaj', 'time': None, 'tags': ['registration', 'last date']}),
    ('Doodh le aana aur gas ka bill bhardena', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['doodh', 'gas', 'bill']}),
    ('Passport renew kardena jaldi', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['passport', 'renew']}),
    ('Doctor appointment book kardena kal ke liye', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['doctor', 'appointment']}),
    ('Notes share kardena sab ko', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['notes', 'share']}),
    ('Paise bhardena account mein', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['paise', 'account']}),
    ('Assignment download kardena portal se aaj', {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'aaj', 'time': None, 'tags': ['assignment', 'download', 'portal']}),
    ('Attendance fill kardena aaj', {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': None, 'tags': ['attendance', 'fill']}),
    ('Sabzi le aana aur bijli ka bill bhardena', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['sabzi', 'bijli', 'bill']}),
    ('Project push kardena GitHub pe kal tak', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'kal', 'time': None, 'tags': ['project', 'GitHub', 'push']}),

    # KAR DENA / BHAR DENA (spaced forms)
    ('Gas ka bill bhar dena', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['gas', 'bill']}),
    ('Light ka bill bhar dena kal tak', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['light', 'bill']}),
    ('Rent bhar dena is mahine', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'is mahine', 'time': None, 'tags': ['rent', 'bill']}),
    ('Fee bhar dena Monday tak', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Monday', 'time': None, 'tags': ['fee']}),
    ('Paise bhar dena account mein aaj', {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': None, 'tags': ['paise', 'account']}),
    ('Assignment submit kar dena aaj raat tak', {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'aaj raat', 'time': None, 'tags': ['assignment', 'submit']}),
    ('Professor ko mail kar dena kal', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['professor', 'mail']}),
    ('Library book return kar dena Friday tak', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Friday', 'time': None, 'tags': ['library', 'book', 'return']}),
    ('Form fill kar dena aaj', {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': None, 'tags': ['form', 'fill']}),
    ('Registration kar dena aaj, last date hai', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'aaj', 'time': None, 'tags': ['registration', 'last date']}),
    ('Notes share kar dena sab ko', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['notes', 'share']}),
    ('Passport renew kar dena jaldi', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['passport', 'renew']}),
    ('Report submit kar dena by 5pm', {'is_reminder': True, 'category': 'DEADLINE', 'date': None, 'time': '5pm', 'tags': ['report', 'submit']}),
    ('Link share kar dena group mein', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['link', 'share']}),
    ('Bijli ka bill bhar dena Sunday se pehle', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Sunday', 'time': None, 'tags': ['bijli', 'bill']}),
    ('Attendance fill kar dena aaj', {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': None, 'tags': ['attendance', 'fill']}),
    ('Project push kar dena GitHub pe kal tak', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'kal', 'time': None, 'tags': ['project', 'GitHub', 'push']}),

    # MAT BHOOLNA / BHOOLNA MAT (dont forget forms)
    ('Kal exam hai, mat bhoolna', {'is_reminder': True, 'category': 'EXAM', 'date': 'kal', 'time': None, 'tags': ['exam', 'reminder']}),
    ('Meeting 3pm pe hai, mat bhoolna', {'is_reminder': True, 'category': 'MEETING', 'date': None, 'time': '3pm', 'tags': ['meeting', 'reminder']}),
    ('Assignment submit karna hai, mat bhoolna', {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': None, 'time': None, 'tags': ['assignment', 'submit']}),
    ('Gas ka bill bharna hai, mat bhoolna', {'is_reminder': True, 'category': 'REMINDER', 'date': None, 'time': None, 'tags': ['gas', 'bill']}),
    ('Kal doctor appointment hai, mat bhoolna 11am', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': '11am', 'tags': ['doctor', 'appointment']}),
    ('Aaj library book return karna hai, mat bhoolna', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'aaj', 'time': None, 'tags': ['library', 'book', 'return']}),
    ('Dadi ka birthday hai kal, mat bhoolna', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['birthday', 'dadi']}),
    ('ID card laana hai kal, mat bhoolna', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['ID card', 'laana']}),
    ('Fee last date hai Friday, mat bhoolna', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Friday', 'time': None, 'tags': ['fee', 'last date']}),
    ('Viva kal 10am hai, mat bhoolna', {'is_reminder': True, 'category': 'EXAM', 'date': 'kal', 'time': '10am', 'tags': ['viva', 'reminder']}),
    ('Bhoolna mat, aaj submission hai', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'aaj', 'time': None, 'tags': ['submission']}),
    ('Bhoolna mat, kal class 8am se hai', {'is_reminder': True, 'category': 'MEETING', 'date': 'kal', 'time': '8am', 'tags': ['class', 'reminder']}),
    ('Bhoolna mat, exam Monday ko hai', {'is_reminder': True, 'category': 'EXAM', 'date': 'Monday', 'time': None, 'tags': ['exam', 'Monday']}),
    ('Bhoolna mat assignment Friday tak submit karna hai', {'is_reminder': True, 'category': 'ASSIGNMENT', 'date': 'Friday', 'time': None, 'tags': ['assignment', 'submit']}),
    ('Bhoolna mat, doctor appointment hai Thursday', {'is_reminder': True, 'category': 'REMINDER', 'date': 'Thursday', 'time': None, 'tags': ['doctor', 'appointment']}),
    ('Bhoolna mat, rent bhar dena kal tak', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'kal', 'time': None, 'tags': ['rent', 'bill']}),
    ('Kal ka plan yaad hai na, bhoolna mat', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['plan', 'reminder']}),
    ('Notes le aana kal, bhoolna mat', {'is_reminder': True, 'category': 'REMINDER', 'date': 'kal', 'time': None, 'tags': ['notes', 'laana']}),
    ('Bill bharna hai Sunday se pehle, bhoolna mat', {'is_reminder': True, 'category': 'DEADLINE', 'date': 'Sunday', 'time': None, 'tags': ['bill', 'bhar dena']}),
    ('Passport copy laani hai aaj, bhoolna mat', {'is_reminder': True, 'category': 'REMINDER', 'date': 'aaj', 'time': None, 'tags': ['passport', 'copy']}),
]

PREFIXES = ['', '', '', 'Important: ', 'Reminder: ', 'Note: ', 'FYI: ', 'Heads up: ']
SUFFIXES = ['', '', '', ' Please note.', ' Sabko yaad dilana tha.', ' Dont miss it.', ' Jaldi karo!']

def augment(msg):
    return random.choice(PREFIXES) + msg + random.choice(SUFFIXES)

random.seed(42)
texts = [make_text(m, r) for m, r in examples_raw]
reminder_pool = [(m, r) for m, r in examples_raw if r['is_reminder']]
noise_pool    = [(m, r) for m, r in examples_raw if not r['is_reminder']]

while len(texts) < 500:
    pool = reminder_pool if random.random() > 0.3 else noise_pool
    m, r = random.choice(pool)
    texts.append(make_text(augment(m), r))

random.shuffle(texts)
dataset = Dataset.from_dict({'text': texts[:500]})
dataset = dataset.train_test_split(test_size=0.1, seed=42)

reminder_count = sum(1 for t in texts[:500] if '"is_reminder": true' in t)
print(f'✅ Dataset ready: {len(texts[:500])} examples')
print(f'   Reminders: {reminder_count} | Noise: {500 - reminder_count}')
print(f'   Train: {len(dataset["train"])} | Val: {len(dataset["test"])}')


## Cell 4 — Load base model + LoRA config

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'google/functiongemma-270m-it'

# 4-bit quantization so it fits on T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

# LoRA — only trains a tiny fraction of weights (~1% of params)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ Model loaded with LoRA adapters')

## Cell 5 — Fine-tune with SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='/kaggle/working/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    max_seq_length=512,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    tokenizer=tokenizer,
)

print('🚀 Starting fine-tuning...')
trainer.train()
print('✅ Fine-tuning complete!')

## Cell 6 — Merge LoRA weights + save full model

In [ ]:
from peft import PeftModel

MERGED_PATH = '/kaggle/working/functiongemma-aloof-merged'

# Merge LoRA adapters back into the base model weights
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print(f'✅ Merged model saved to {MERGED_PATH}')

## Cell 7 — Convert to .tflite (LiteRT)

In [ ]:
# Install conversion tools (CPU only — GPU not needed here)
!pip install -q 'litert-torch>=0.8.0'

import os
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.utilities import converter
from litert_torch.generative.utilities.export_config import ExportConfig
from litert_torch.generative.layers import kv_cache

TFLITE_OUTPUT = '/kaggle/working/tflite_output'
os.makedirs(TFLITE_OUTPUT, exist_ok=True)

print('⏳ Converting to TFLite (this takes 5-10 mins)...')

pytorch_model = gemma3.build_model_270m(MERGED_PATH)

export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

converter.convert_to_tflite(
    pytorch_model,
    output_path=TFLITE_OUTPUT,
    output_name_prefix='functiongemma-aloof',
    prefill_seq_len=512,
    kv_cache_max_len=1024,
    quantize='dynamic_int8',
    export_config=export_config,
)

tflite_file = os.path.join(TFLITE_OUTPUT, 'functiongemma-aloof.tflite')
print(f'✅ TFLite model saved: {tflite_file}')

## Cell 8 — Bundle into .task file (MediaPipe)

In [ ]:
!pip install -q mediapipe

from mediapipe.tasks.python.genai import bundler

TASK_OUTPUT = '/kaggle/working/functiongemma-aloof.task'

config = bundler.BundleConfig(
    tflite_model=tflite_file,
    tokenizer_model=f'{MERGED_PATH}/tokenizer.model',
    start_token='<bos>',
    stop_tokens=['<eos>', '<end_of_turn>'],
    output_filename=TASK_OUTPUT,
    prompt_prefix='<start_of_turn>user\n',
    prompt_suffix='<end_of_turn>\n<start_of_turn>model\n',
)

bundler.create_bundle(config)

size_mb = os.path.getsize(TASK_OUTPUT) / (1024 * 1024)
print(f'✅ .task file created: {TASK_OUTPUT}')
print(f'   Size: {size_mb:.1f} MB')

## Cell 9 — Upload to your HuggingFace repo

In [ ]:
from huggingface_hub import HfApi

# ⚠️ Replace with your HuggingFace username
HF_USERNAME  = 'your-hf-username'
HF_REPO_NAME = 'functiongemma-aloof'
REPO_ID = f'{HF_USERNAME}/{HF_REPO_NAME}'

api = HfApi()

# Create repo if it doesn't exist
api.create_repo(repo_id=REPO_ID, exist_ok=True, token=hf_token)

# Upload the .task file
api.upload_file(
    path_or_fileobj=TASK_OUTPUT,
    path_in_repo='functiongemma-aloof.task',
    repo_id=REPO_ID,
    token=hf_token,
)

download_url = f'https://huggingface.co/{REPO_ID}/resolve/main/functiongemma-aloof.task'
print(f'✅ Uploaded to HuggingFace!')
print(f'\n🔗 Your download URL (copy this into ModelDownloadManager.kt):')
print(f'   {download_url}')